# Module 3 — Text mining : les concepts

**CONSEIL :** Sauvegarde une copie de ce notebook dans ton Drive avant de commencer !

**Objectifs d'apprentissage**
- **Collecter** un corpus de texte en ligne par scraping (`requests` + `BeautifulSoup`)
- Passer d'un texte brut à des **tokens** propres (nettoyage, stopwords, stemming)
- Transformer du texte en nombres : **bag of words**, **term-document matrix**, **TF-IDF**
- Comprendre le **principe d'embedding** : des vecteurs creux aux vecteurs denses
- Voir **Word2Vec** et **BERT** à l'œuvre, et mesurer une **similarité cosinus**
- Utiliser tout ça pour **tester des hypothèses de recherche** sur un vrai corpus

## La problématique : quels imaginaires du voyage sur Airbnb ?

Les plateformes comme Airbnb ne vendent pas seulement un hébergement, mais aussi une certaine **vision du voyage**. Certaines annonces insistent sur le confort, les équipements, le luxe du logement ; d'autres mettent en avant l'authenticité, la découverte du quartier, l'expérience locale.

On va analyser les **descriptions d'annonces** de trois villes (Bruxelles, Paris, Amsterdam, données [Inside Airbnb](https://insideairbnb.com)) pour comprendre quels discours les hôtes mobilisent. Fil rouge du notebook : transformer ces descriptions en vecteurs, construire des **requêtes thématiques** et mesurer leur proximité avec chaque annonce pour tester trois hypothèses :

- **H1** — les logements les plus **chers** sont davantage associés à un discours de **confort / luxe** ;
- **H2** — les logements en quartier **central** sont davantage associés à un discours d'**expérience locale** ;
- **H3** — les **chambres privées** mettent davantage en avant l'**hospitalité** et la rencontre.

Chaque étape technique du text mining nous rapproche de la réponse.

## Setup

In [ ]:
# Sur Colab, décommente la ligne suivante pour installer les dépendances :
# !pip install -q requests beautifulsoup4 scikit-learn gensim sentence-transformers wordcloud matplotlib pandas

import os
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

## 3.1 Collecter : aller chercher les données en ligne (scraping)

Première étape de tout projet de text mining : **récupérer un corpus**. Ici les données vivent sur le site [Inside Airbnb](https://insideairbnb.com/get-the-data/), qui publie pour chaque ville un fichier `listings.csv.gz`.

Le scraping se fait en deux temps :
1. `requests` télécharge le **HTML** d'une page web ;
2. `BeautifulSoup` **parse** ce HTML pour en extraire ce qui nous intéresse (ici, les liens de téléchargement).

> **Bonnes pratiques** : on lit les conditions d'utilisation et le `robots.txt`, on s'identifie via un `User-Agent`, et on espace les requêtes pour ne pas surcharger le serveur.

In [ ]:
DATA_PAGE = "https://insideairbnb.com/get-the-data/"
headers = {"User-Agent": "Mozilla/5.0 (cours text mining)"}

html = requests.get(DATA_PAGE, headers=headers, timeout=60).text
soup = BeautifulSoup(html, "html.parser")

# On parcourt toutes les balises <a> et on garde les liens vers un listings.csv.gz
liens = [a["href"] for a in soup.find_all("a", href=True)
         if a["href"].endswith("listings.csv.gz")]

print(f"{len(liens)} liens de villes trouvés sur la page. Exemples :")
for lien in liens[:4]:
    print(" -", lien)

Une fois un lien obtenu, on télécharge le fichier avec `requests` et on le lit avec pandas (`pd.read_csv(..., compression="gzip")`). Ces fichiers sont **lourds** (Paris ≈ 80 000 annonces), alors on construit un **échantillon léger** : 400 annonces par ville, descriptions **en anglais** (les hôtes visent une clientèle internationale), prix renseigné.

La fonction ci-dessous fait tout, **directement depuis Inside Airbnb** : aucun fichier à uploader. (Si une copie locale ou hébergée existe déjà, on la réutilise pour aller plus vite.)

In [ ]:
import io, html

EN = {"the","and","with","you","your","for","this","is","are","of","to","in","near","from","we","our"}
FR = {"le","la","les","et","avec","vous","votre","pour","une","un","des","du","est","dans","près","nous"}

def en_anglais(texte):
    mots = re.findall(r"[a-zà-ÿ]+", texte.lower())
    en = sum(m in EN for m in mots); fr = sum(m in FR for m in mots)
    return en >= 3 and en > fr * 1.5

def nettoyer(texte):
    texte = re.sub(r"<[^>]+>", " ", html.unescape(str(texte)))
    return re.sub(r"\s+", " ", texte).strip()

def construire_corpus(n_par_ville=400):
    villes = {"Brussels": "/brussels/", "Paris": "/paris/", "Amsterdam": "/amsterdam/"}
    # Le dernier dump Paris a des prix vides : on épingle une date connue-bonne.
    epingles = {"Paris": "https://data.insideairbnb.com/france/ile-de-france/paris/2025-06-06/data/listings.csv.gz"}
    cols = ["id","description","neighbourhood_cleansed","room_type","price","latitude","longitude"]
    frames = []
    for ville, jeton in villes.items():
        url = epingles.get(ville) or next(l for l in liens if jeton in l)
        brut = pd.read_csv(io.BytesIO(requests.get(url, headers=headers, timeout=180).content),
                           compression="gzip")[cols].dropna(subset=["description"])
        brut["description"] = brut["description"].map(nettoyer)
        brut = brut[brut["description"].str.split().str.len() >= 15]
        brut = brut[brut["description"].map(en_anglais)]
        brut["price"] = pd.to_numeric(brut["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
                                      errors="coerce")
        brut = brut.dropna(subset=["price"])
        brut["city"] = ville
        frames.append(brut.sample(min(n_par_ville, len(brut)), random_state=42))
    return (pd.concat(frames, ignore_index=True)
              .rename(columns={"neighbourhood_cleansed": "neighbourhood"}))

# Chemin rapide (copie locale ou hébergée) sinon reconstruction depuis la source.
LOCAL = "../../../data/airbnb/airbnb_listings.csv"
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/airbnb/airbnb_listings.csv"
try:
    annonces = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
except Exception:
    print("Construction du corpus directement depuis Inside Airbnb...")
    annonces = construire_corpus()

print(f"{len(annonces)} annonces — {annonces['city'].value_counts().to_dict()}")
annonces[["city", "neighbourhood", "room_type", "price", "description"]].head(3)

## 3.2 Tokenization et nettoyage

Un ordinateur ne « lit » pas une phrase : il manipule des **tokens** (des mots, ou des morceaux de mots). Avant toute analyse, on transforme chaque description en une liste de tokens propres :

- tout en **minuscules** (`Cozy` et `cozy` doivent être le même mot) ;
- on retire la **ponctuation** et les chiffres ;
- on enlève les **stopwords** (mots outils très fréquents et peu informatifs : *the*, *a*, *and*…).

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(texte):
    texte = texte.lower()
    mots = re.findall(r"[a-z]+", texte)             # suites de lettres uniquement
    return [m for m in mots if m not in ENGLISH_STOP_WORDS and len(m) > 2]

exemple = annonces["description"].iloc[0]
print("Texte brut :\n", exemple[:200], "...\n")
print("Tokens nettoyés :\n", tokenize(exemple)[:20])

### Stemming vs lemmatisation

`apartment` et `apartments`, `located` et `location`… désignent presque la même chose. Deux familles de techniques **regroupent les variantes d'un mot** :

- le **stemming** coupe les suffixes de façon mécanique (`apartments` → `apart`). Rapide, brutal, le résultat n'est pas toujours un vrai mot.
- la **lemmatisation** ramène chaque mot à sa forme de dictionnaire (`better` → `good`, `was` → `be`). Plus fin, mais demande un modèle linguistique.

In [ ]:
from gensim.parsing.porter import PorterStemmer
stem = PorterStemmer().stem

for mot in ["cozy", "located", "apartments", "renovated", "welcoming", "neighbourhood"]:
    print(f"{mot:>14} -> {stem(mot)}")

### Hack Time

- Applique `tokenize` à une annonce d'Amsterdam et compare les tokens à ceux de Bruxelles.
- La liste `ENGLISH_STOP_WORDS` te paraît-elle complète ? Ajoute tes propres stopwords (par ex. `room`, `apartment`) et regarde l'effet.

In [ ]:
# Votre code ici

## 3.3 Bag of words et term-document matrix

L'approche la plus simple pour transformer un corpus en nombres : le **bag of words** (« sac de mots »). On oublie l'ordre des mots et on ne retient que **combien de fois** chaque mot apparaît dans chaque document.

En empilant tous les documents, on obtient la **term-document matrix** (TDM) : une ligne par annonce, une colonne par mot du vocabulaire, et dans chaque case un comptage. `CountVectorizer` fait tout le travail (tokenization + stopwords + comptage).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", min_df=5, max_df=0.5)
X_counts = vectorizer.fit_transform(annonces["description"])

print("Matrice :", X_counts.shape, "(annonces × mots du vocabulaire)")
densite = X_counts.nnz / (X_counts.shape[0] * X_counts.shape[1])
print(f"Cases non nulles : {densite:.2%}  ->  la matrice est très CREUSE (sparse)")

# Aperçu : comptes de quelques mots sur 5 annonces
mots = ["apartment", "cozy", "luxury", "metro", "quiet", "central"]
presents = [m for m in mots if m in vectorizer.vocabulary_]
idx = [vectorizer.vocabulary_[m] for m in presents]
pd.DataFrame(X_counts[:5, idx].toarray(), columns=presents)

## 3.4 Réduction de dimension

Le vocabulaire compte des milliers de mots : la TDM a donc des **milliers de colonnes**, presque toutes à zéro. C'est encombrant et bruyant. On réduit la dimension de deux façons :

1. **filtrer** le vocabulaire (`min_df` : ignorer les mots trop rares ; `max_df` : ignorer les mots trop fréquents) — déjà fait ci-dessus ;
2. **projeter** la matrice dans un petit nombre de dimensions avec une SVD (`TruncatedSVD`), pour pouvoir la **visualiser** en 2D.

On colore chaque annonce par sa ville : voit-on apparaître une structure ?

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5)
X_tfidf = tfidf.fit_transform(annonces["description"])

coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_tfidf)

plt.figure(figsize=(7, 5))
for ville, couleur in zip(["Brussels", "Paris", "Amsterdam"],
                          ["#1f77b4", "#d62728", "#2ca02c"]):
    m = (annonces["city"] == ville).values
    plt.scatter(coords[m, 0], coords[m, 1], s=10, alpha=0.5, label=ville, color=couleur)
plt.legend(); plt.title("Annonces projetées en 2D (TF-IDF + SVD)")
plt.xlabel("composante 1"); plt.ylabel("composante 2"); plt.show()

## 3.5 TF-IDF : faire ressortir les mots distinctifs

Avec un simple comptage, les mots les plus fréquents (`apartment`, `room`…) écrasent tout. Le **TF-IDF** (*Term Frequency – Inverse Document Frequency*) pondère chaque mot par sa **rareté** : un mot fréquent dans une annonce mais rare dans le corpus reçoit un poids fort. C'est ce qui fait ressortir le vocabulaire **distinctif**.

On calcule, ville par ville, les mots au TF-IDF moyen le plus élevé.

In [ ]:
vocab = np.array(tfidf.get_feature_names_out())

def mots_distinctifs(masque, k=10):
    moyenne = np.asarray(X_tfidf[masque].mean(axis=0)).ravel()
    top = moyenne.argsort()[::-1][:k]
    return ", ".join(vocab[top])

for ville in ["Brussels", "Paris", "Amsterdam"]:
    print(f"{ville:>10} : {mots_distinctifs((annonces['city'] == ville).values)}")

### Hack Time

- Calcule les mots distinctifs non plus par ville, mais par **type de logement** (`room_type`).
- Une chambre privée (`Private room`) utilise-t-elle un vocabulaire différent d'un logement entier ?

In [ ]:
# Votre code ici
# Indice : remplace le masque par (annonces['room_type'] == 'Private room').values

## 3.6 Le principe d'embedding

TF-IDF reste une représentation **creuse** : chaque mot est une colonne isolée. Le modèle ne sait pas que `cosy`, `comfortable` et `cozy` parlent de la même chose — pour lui, ce sont trois colonnes sans rapport.

L'idée de l'**embedding** : représenter chaque mot (ou chaque document) par un **vecteur dense** de quelques centaines de nombres, appris de telle sorte que **deux mots de sens proche aient des vecteurs proches**. On passe d'une matrice creuse de milliers de colonnes à un espace dense où la **distance capture le sens**.

Deux générations de modèles :
- **Word2Vec** (2013) : un vecteur par mot, appris à partir des contextes ;
- **BERT** (2018) : des vecteurs **contextuels**, calculés pour une phrase entière.

## 3.7 Word2Vec : apprendre le sens des mots par leur contexte

Word2Vec apprend qu'un mot se définit par les mots qui l'entourent. On l'entraîne **sur notre propre corpus** d'annonces, puis on demande les mots les plus proches dans l'espace appris.

> Notre corpus est petit (1 200 annonces) : les voisins sont indicatifs, pas parfaits. Sur des milliards de mots, Word2Vec capture des analogies fines (*king − man + woman ≈ queen*).

In [ ]:
from gensim.models import Word2Vec

phrases = [tokenize(t) for t in annonces["description"]]
w2v = Word2Vec(phrases, vector_size=80, window=5, min_count=5,
               workers=2, epochs=40, seed=42)

print("Taille du vocabulaire appris :", len(w2v.wv), "mots\n")
for mot in ["luxury", "cozy", "metro"]:
    if mot in w2v.wv:
        voisins = ", ".join(w for w, _ in w2v.wv.most_similar(mot, topn=5))
        print(f"Proches de '{mot}' : {voisins}")

## 3.8 BERT + similarité cosinus : tester nos hypothèses

Pour comparer une **annonce** à une **requête thématique** (« un logement luxueux »), on a besoin d'un vecteur qui capture le **sens de la phrase entière**, pas mot à mot. C'est ce que produit un modèle de type **BERT**, via la librairie `sentence-transformers`.

La recette :
1. on encode chaque annonce et chaque requête en un vecteur dense ;
2. on mesure leur proximité par la **similarité cosinus** (1 = sens très proche, 0 = sans rapport) ;
3. on croise ces scores avec le prix, la centralité, le type de logement → **H1, H2, H3**.

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
emb_annonces = model.encode(annonces["description"].tolist(),
                            show_progress_bar=False, batch_size=64)

requetes = {
    "confort_luxe": "a luxurious, comfortable and well-equipped home",
    "experience_locale": "an authentic local experience to explore the neighbourhood",
    "hospitalite": "a welcoming host, hospitality and meeting local people",
}
emb_requetes = model.encode(list(requetes.values()))

sims = util.cos_sim(emb_annonces, emb_requetes).numpy()
for i, nom in enumerate(requetes):
    annonces[f"sim_{nom}"] = sims[:, i]

annonces[["city", "price", "room_type",
          "sim_confort_luxe", "sim_experience_locale", "sim_hospitalite"]].head()

### H1 — Prix élevé ↔ discours de confort / luxe

On découpe les annonces en quatre tranches de prix et on regarde la similarité moyenne avec la requête « confort / luxe ».

In [ ]:
annonces["tranche_prix"] = pd.qcut(annonces["price"], 4,
                                   labels=["bas", "moyen", "élevé", "très élevé"],
                                   duplicates="drop")
h1 = annonces.groupby("tranche_prix", observed=True)["sim_confort_luxe"].mean()
print(h1.round(3))
h1.plot(kind="bar", rot=0, ylabel="similarité moyenne",
        title="H1 — discours confort/luxe selon le prix"); plt.show()

### H2 — Quartier central ↔ discours d'expérience locale

On calcule la distance de chaque logement au centre de sa ville (à partir de la latitude/longitude), puis on compare les annonces **centrales** (plus proches que la médiane de leur ville) aux **périphériques**.

In [ ]:
centres = {"Brussels": (50.8467, 4.3525),
           "Paris": (48.8566, 2.3522),
           "Amsterdam": (52.3676, 4.9041)}

def distance_centre(row):
    clat, clon = centres[row["city"]]
    return np.hypot(row["latitude"] - clat, row["longitude"] - clon)

annonces["dist_centre"] = annonces.apply(distance_centre, axis=1)
annonces["central"] = annonces.groupby("city")["dist_centre"].transform(
    lambda d: d < d.median())

h2 = annonces.groupby("central")["sim_experience_locale"].mean()
h2.index = ["périphérique", "central"]
print(h2.round(3))
h2.plot(kind="bar", rot=0, ylabel="similarité moyenne",
        title="H2 — discours d'expérience locale selon la centralité"); plt.show()

### H3 — Chambre privée ↔ discours d'hospitalité

On compare le discours d'hospitalité entre les logements entiers et les chambres privées (où l'hôte partage souvent son logement).

In [ ]:
sous = annonces[annonces["room_type"].isin(["Entire home/apt", "Private room"])]
h3 = sous.groupby("room_type")["sim_hospitalite"].mean()
print(h3.round(3))
h3.plot(kind="bar", rot=0, ylabel="similarité moyenne",
        title="H3 — discours d'hospitalité selon le type de logement"); plt.show()

### Hack Time

- Invente une **nouvelle requête thématique** (par ex. `"a quiet place to work remotely"`), encode-la et calcule sa similarité avec chaque annonce.
- Quelle ville obtient les scores les plus élevés ? Le résultat te paraît-il crédible ?

In [ ]:
# Votre code ici

### Interpréter, avec prudence

Sur notre échantillon, **H3** ressort nettement (les chambres privées parlent plus d'hospitalité), **H1** est faiblement soutenue, **H2** ne l'est pas. Les écarts sont parfois ténus : petit corpus, requêtes volontairement génériques, et un seul modèle d'embedding. C'est tout l'intérêt de la démarche — affiner les requêtes, agrandir le corpus, croiser les modèles — plutôt que de conclure trop vite. Le text mining **outille** l'analyse, il ne remplace pas l'interprétation du chercheur.

## Récap module 3

| Méthode | Représentation | Capture le sens ? | Ce qu'on en fait |
|---|---|---|---|
| Bag of words / TDM | creuse (comptages) | non | compter, repérer le vocabulaire |
| TF-IDF | creuse (pondérée) | non | mots **distinctifs** |
| Word2Vec | dense (par mot) | un peu | mots proches |
| BERT (`sentence-transformers`) | dense (par phrase) | oui | **similarité** annonce ↔ requête, test d'hypothèses |

On est passé d'un texte brut récupéré en ligne à des **vecteurs** qui répondent à une vraie question de recherche.

→ Direction le **notebook 2** : cinq **applications** concrètes du text mining (nuage de mots, sentiment, classification, recherche sémantique, biais), sur un corpus de critiques de films.